# 📘 Fincept Notebook — Time Value of Money

**Finance · Beginner · ~10 min · Standard library only**

Money today is worth more than the same money tomorrow. This notebook walks through the core building blocks of valuation: discounting and compounding a single cash flow, valuing a stream of cash flows with NPV, solving for the internal rate of return (IRR), and breaking a loan into its amortization schedule.

**What you'll learn**
- Present value (PV) and future value (FV), and how compounding frequency changes the answer
- Net present value (NPV) of a cash-flow stream at a given discount rate
- How to solve for IRR numerically with the bisection method
- How to build a loan amortization schedule


## 1. Present value & future value

A single cash flow `C` grows to a **future value** `FV = C · (1 + r/m)^(m·t)` and a future amount discounts back to a **present value** `PV = C / (1 + r/m)^(m·t)`, where `r` is the annual rate, `m` is the compounding frequency per year, and `t` is years.

The table below shows how $1,000 grows over time and how the discount rate changes today's value of a fixed future amount.


In [ ]:
import math

def future_value(pv, annual_rate, years, m=1):
    """Future value of a lump sum with m compounding periods per year."""
    return pv * (1 + annual_rate / m) ** (m * years)

def present_value(fv, annual_rate, years, m=1):
    """Present value of a future lump sum."""
    return fv / (1 + annual_rate / m) ** (m * years)

# --- Worked example -------------------------------------------------
principal = 1000.0
rate = 0.06          # 6% annual
years = 5
fv_annual = future_value(principal, rate, years, m=1)
print(f"Invest ${principal:,.2f} at {rate:.1%} for {years} years (annual compounding):")
print(f"  Future value = ${fv_annual:,.2f}")
print(f"  Check PV     = ${present_value(fv_annual, rate, years, m=1):,.2f}  (should return the principal)")

# --- Compounding frequency matters ----------------------------------
print("\nEffect of compounding frequency ($1,000 at 6% for 5 years):")
print(f"{'Frequency':<14}{'m':>4}{'Future value':>16}")
print('-' * 34)
for label, m in [('Annual', 1), ('Semiannual', 2), ('Quarterly', 4), ('Monthly', 12), ('Daily', 365)]:
    print(f"{label:<14}{m:>4}{future_value(principal, rate, years, m):>16,.2f}")

# --- FV grid over several years and rates ---------------------------
rates = [0.03, 0.05, 0.07, 0.10]
horizons = [1, 3, 5, 10, 20]
print("\nFuture value of $1,000 (annual compounding):")
header = f"{'Years':>6}" + ''.join(f"{r:>12.0%}" for r in rates)
print(header)
print('-' * len(header))
for t in horizons:
    row = f"{t:>6}" + ''.join(f"{future_value(principal, r, t):>12,.0f}" for r in rates)
    print(row)

# --- PV of a fixed future amount at several discount rates ----------
target = 10000.0
print(f"\nPresent value of ${target:,.0f} received in 5 years:")
print(f"{'Discount rate':>14}{'Present value':>16}")
print('-' * 30)
for r in rates:
    print(f"{r:>14.0%}{present_value(target, r, 5):>16,.2f}")


## 2. Net present value of a cash-flow stream

A project's **NPV** discounts every future cash flow back to today and sums them:

`NPV = Σ  Cₜ / (1 + r)ᵗ`  for t = 0, 1, 2, …

Year 0 is the up-front outlay (a negative number). A positive NPV means the project adds value at the chosen discount rate. Below we value a project with an initial $10,000 investment followed by five years of inflows, and tabulate how NPV falls as the discount rate rises.


In [ ]:
def npv(rate, cashflows):
    """NPV of cashflows[0..n] where index = period (0 = today)."""
    return sum(cf / (1 + rate) ** t for t, cf in enumerate(cashflows))

# Initial outlay then five annual inflows
cashflows = [-10000, 2500, 3000, 3500, 4000, 4500]
discount_rate = 0.08

print("Project cash flows:")
print(f"{'Year':>5}{'Cash flow':>14}{'Discount factor':>18}{'Present value':>16}")
print('-' * 53)
for t, cf in enumerate(cashflows):
    df = 1 / (1 + discount_rate) ** t
    print(f"{t:>5}{cf:>14,.2f}{df:>18.4f}{cf * df:>16,.2f}")
print('-' * 53)
print(f"{'NPV at ' + format(discount_rate, '.0%'):>23}{'':>18}{npv(discount_rate, cashflows):>16,.2f}")

# NPV at a range of discount rates
print("\nNPV sensitivity to the discount rate:")
print(f"{'Discount rate':>14}{'NPV':>16}")
print('-' * 30)
for r in [0.00, 0.04, 0.08, 0.12, 0.16, 0.20]:
    print(f"{r:>14.0%}{npv(r, cashflows):>16,.2f}")


## 3. Internal rate of return (IRR) by bisection

The **IRR** is the discount rate that makes NPV exactly zero — the project's break-even rate of return. There is no closed-form solution in general, so we find it numerically. The **bisection method** brackets the IRR between a low and high rate (where NPV changes sign) and repeatedly halves the interval until it converges.


In [ ]:
def npv(rate, cashflows):
    return sum(cf / (1 + rate) ** t for t, cf in enumerate(cashflows))

def irr_bisection(cashflows, low=-0.9, high=1.0, tol=1e-9, max_iter=200):
    """Find IRR via bisection. Requires NPV(low) and NPV(high) to differ in sign."""
    f_low, f_high = npv(low, cashflows), npv(high, cashflows)
    if f_low * f_high > 0:
        raise ValueError("NPV does not change sign on the bracket; widen [low, high].")
    for _ in range(max_iter):
        mid = (low + high) / 2
        f_mid = npv(mid, cashflows)
        if abs(f_mid) < tol:
            return mid
        if f_low * f_mid < 0:
            high, f_high = mid, f_mid
        else:
            low, f_low = mid, f_mid
    return (low + high) / 2

# Same project as before (re-embedded so this cell stands alone)
cashflows = [-10000, 2500, 3000, 3500, 4000, 4500]

irr = irr_bisection(cashflows)
print(f"IRR = {irr:.6f}  ({irr:.2%})")
print(f"Verification: NPV at the IRR = {npv(irr, cashflows):,.6f}  (should be ~0)")

print("\nDecision rule: accept the project when IRR exceeds your cost of capital.")
for hurdle in [0.06, 0.08, 0.12, 0.15]:
    verdict = 'ACCEPT' if irr > hurdle else 'REJECT'
    print(f"  Hurdle rate {hurdle:>5.0%}  ->  {verdict}")


## 4. Loan amortization schedule

A fully-amortizing loan is repaid in equal periodic payments. Each payment covers the period's interest first, and whatever is left reduces the principal. The level payment is:

`PMT = P · i / (1 − (1 + i)^(−n))`

where `P` is the principal, `i` is the per-period rate, and `n` is the number of payments. As the balance shrinks, the interest portion falls and the principal portion grows.


In [ ]:
def level_payment(principal, annual_rate, years, periods_per_year=12):
    """Equal periodic payment for a fully-amortizing loan."""
    i = annual_rate / periods_per_year
    n = years * periods_per_year
    if i == 0:
        return principal / n
    return principal * i / (1 - (1 + i) ** -n)

# A $25,000 loan at 7.5% APR over 2 years, monthly payments
principal = 25000.0
annual_rate = 0.075
years = 2
ppy = 12
i = annual_rate / ppy
n = years * ppy
pmt = level_payment(principal, annual_rate, years, ppy)

print(f"Loan: ${principal:,.2f} at {annual_rate:.2%} APR, {n} monthly payments")
print(f"Level monthly payment = ${pmt:,.2f}\n")

print(f"{'Pmt':>4}{'Payment':>12}{'Interest':>12}{'Principal':>12}{'Balance':>14}")
print('-' * 54)
balance = principal
total_interest = 0.0
for k in range(1, n + 1):
    interest = balance * i
    principal_paid = pmt - interest
    balance -= principal_paid
    if k == n:  # absorb tiny rounding drift in the final payment
        principal_paid += balance
        balance = 0.0
    total_interest += interest
    print(f"{k:>4}{pmt:>12,.2f}{interest:>12,.2f}{principal_paid:>12,.2f}{balance:>14,.2f}")
print('-' * 54)
print(f"Total paid     = ${pmt * n:,.2f}")
print(f"Total interest = ${total_interest:,.2f}")


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
